# Week 03 — Type hints, defensive programming, logging and pipelines

**Seismic Python & AI Engineering | Hands-on notebook**

Scientific software often fails at its boundaries: empty arrays, inconsistent units, silent NaNs and unclear error messages. This week makes these boundaries explicit.

**Estimated independent study:** 4–5 hours in this notebook, plus 2–3 hours on repository exercises and review.  
**Data:** entirely synthetic; no external API, proprietary dataset, or GPU required.  
**Prerequisite:** complete the root README setup and open Jupyter from the repository root.

### Learning outcomes
- Design validation checks and intentional exception messages.
- Distinguish warnings, logs, exceptions and assertions.
- Use dataclasses and typed configuration to avoid magic numbers.
- Build a simple reproducible analysis pipeline.

### How to study
Read the explanations, execute the worked cells in order, complete the **Your turn** tasks, and implement the week's function in `src/seismic_toolkit/challenges.py`. Run the targeted exercise test to assess progress. The initial baseline tests should pass even before you complete the challenges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from seismic_toolkit import __version__
print(f"Seismic toolkit version: {__version__}")
np.set_printoptions(precision=4, suppress=True)

## 1. Defensive programming for scientific arrays
Check dimensions, finiteness, sampling interval and units at **module boundaries**. A silent NaN can contaminate an entire evaluation; a hidden change from milliseconds to seconds can make a physical result appear plausible while being wrong.

Python type hints are helpful documentation and support static analysis, but **do not** validate array dimensions or numeric ranges at runtime.

In [ ]:
from seismic_toolkit.validation import require_trace, require_positive

cases = [np.array([0., 1., -1.]), np.array([1., np.nan]), np.ones((2, 2)), []]
for value in cases:
    try:
        x = require_trace(value)
        print("Valid trace:", x.shape)
    except ValueError as exc:
        print("Rejected:", exc)

## 2. Structured configurations with dataclasses
A configuration object names parameters, supplies defaults and enables validation. It is more readable than scattering hard-coded values through multiple scripts. Dataclasses are ideal for plain data; they are not a requirement for every program.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class AnalysisConfig:
    dt_s: float = 0.001
    max_lag_s: float = 0.016
    peak_frequency_hz: float = 30.0

    def __post_init__(self):
        require_positive(self.dt_s, "dt_s")
        require_positive(self.peak_frequency_hz, "peak_frequency_hz")
        if self.max_lag_s < 0:
            raise ValueError("max_lag_s must not be negative")

config = AnalysisConfig()
print(config)

## 3. Logging is not the same as printing
`print` is fine in notebooks. For multi-module software, use the `logging` module so operational users can control severity, context and output destination. Never log secrets or restricted data. Use exceptions for errors that prevent correct execution; use logs for diagnostic context.

In [ ]:
import logging
logger = logging.getLogger("course.week03")
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())
logger.setLevel(logging.INFO)
logger.info("Starting synthetic validation demo")
from seismic_toolkit.pipeline import run_synthetic_demo
result = run_synthetic_demo(dt=config.dt_s, delay_samples=4)
print({k: v for k, v in result.items() if not isinstance(v, np.ndarray)})

## 4. Pure functions and reproducibility
A pure numerical function returns the same output for the same inputs and does not mutate hidden external state. Use `np.random.default_rng(seed)` when random values are required; document seeds, units, package versions, train/test definitions and data provenance.

**Your turn A:** rerun the same synthetic pipeline twice. Confirm exact array equality, then change the sample interval and explain which numerical quantities should change.

In [ ]:
a = run_synthetic_demo(dt=0.001, delay_samples=5)
b = run_synthetic_demo(dt=0.001, delay_samples=5)
np.testing.assert_array_equal(a["monitor"], b["monitor"])
assert np.isclose(a["estimated_delay_s"], 0.005)
print("Deterministic replay: PASS")

## 5. Independent implementation — Week 3 challenge
Implement `moving_rms(trace, window)` in `challenges.py`.

Contract: `window` is a **positive odd integer**; each output sample is the RMS over its centered window; at the edges, clip the window to available samples rather than padding with invented values. Output length equals input length. Invalid windows raise `ValueError`.

**Your turn B:** compare your output to an intentionally slow Python loop on a five-element toy trace. Only consider vectorization after the reference solution is correct.

In [ ]:
toy = np.array([1., 2., 3., 4., 5.])
expected_center = np.sqrt((2**2 + 3**2 + 4**2) / 3)
expected_left_edge = np.sqrt((1**2 + 2**2) / 2)
print("Center value expected:", round(expected_center, 5))
print("Left edge expected:", round(expected_left_edge, 5))

## 6. Applied problem: build a reproducible CLI
The repository includes `seismic-demo` (from the `pyproject.toml` entry point) and `python -m seismic_toolkit.cli`.

**Your turn C:** inspect `cli.py`. Add an optional `--output` path that writes only JSON-serializable summary metadata, never entire raw traces by default. Document invalid CLI inputs and confirm the program exits with a useful error. Remember: user input is not trusted.

In [ ]:
import subprocess, sys
completed = subprocess.run(
    [sys.executable, "-m", "seismic_toolkit.cli", "--delay-samples", "4"],
    capture_output=True, text=True, check=True,
)
print(completed.stdout)

---

## Weekly submission and self-review

1. Complete the independent exercise in `src/seismic_toolkit/challenges.py`.
2. Run: `python -m pytest -o addopts='' -m exercise -k week3 -q`.
3. Run baseline regression tests: `python -m pytest -q`.
4. Commit a focused change to a feature branch; document what changed and why.
5. Record one failure mode, one design decision, and one next-step question in your learning journal.

**Further work:** see [`../exercises/week03.md`](../exercises/week03.md) for a graded problem, extra practice and acceptance criteria. Compare with `solutions/reference_challenges.py` only **after** attempting your own solution.

**Important:** all datasets and algorithms are educational demonstrations, not validated reservoir-interpretation systems.